### DATA ACCESS USING APP

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
spark.conf.set("fs.azure.account.auth.type.datalakeadventure.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.datalakeadventure.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.datalakeadventure.dfs.core.windows.net", "<>")
spark.conf.set("fs.azure.account.oauth2.client.secret.datalakeadventure.dfs.core.windows.net", "<>")
spark.conf.set("fs.azure.account.oauth2.client.endpoint.datalakeadventure.dfs.core.windows.net", "https://login.windows.net/<>/oauth2/token")

### DATA LOADING

### read data

In [0]:
df_calender = spark.read.format('csv')\
    .option('header',True)\
    .option('inferSchema', True)\
    .load('abfss://bronze@datalakeadventure.dfs.core.windows.net/AdventureWorks_Calendar/AdventureWorks_Calendar.csv')

In [0]:
df_Customers = spark.read.format('csv')\
    .option('header',True)\
    .option('inferSchema', True)\
    .load('abfss://bronze@datalakeadventure.dfs.core.windows.net/AdventureWorks_Customers/AdventureWorks_Customers.csv')

In [0]:
df_Product_Categories = spark.read.format('csv')\
    .option('header',True)\
    .option('inferSchema', True)\
    .load('abfss://bronze@datalakeadventure.dfs.core.windows.net/AdventureWorks_Product_Categories/AdventureWorks_Product_Categories.csv')

In [0]:
df_Product_Subcategories = spark.read.format('csv')\
    .option('header',True)\
    .option('inferSchema', True)\
    .load('abfss://bronze@datalakeadventure.dfs.core.windows.net/AdventureWorks_Product_Subcategories/AdventureWorks_Product_Subcategories.csv')

In [0]:
df_Products = spark.read.format('csv')\
    .option('header',True)\
    .option('inferSchema', True)\
    .load('abfss://bronze@datalakeadventure.dfs.core.windows.net/AdventureWorks_Products/AdventureWorks_Products.csv')

In [0]:
df_Returns = spark.read.format('csv')\
    .option('header',True)\
    .option('inferSchema', True)\
    .load('abfss://bronze@datalakeadventure.dfs.core.windows.net/AdventureWorks_Returns/AdventureWorks_Returns.csv')

In [0]:
df_Sales_2015 = spark.read.format('csv')\
    .option('header',True)\
    .option('inferSchema', True)\
    .load('abfss://bronze@datalakeadventure.dfs.core.windows.net/AdventureWorks_Sales_2015/AdventureWorks_Sales_2015.csv')

In [0]:
df_Sales_2016 = spark.read.format('csv')\
    .option('header',True)\
    .option('inferSchema', True)\
    .load('abfss://bronze@datalakeadventure.dfs.core.windows.net/AdventureWorks_Sales_2016/AdventureWorks_Sales_2016.csv')

In [0]:
df_Sales_2017 = spark.read.format('csv')\
    .option('header',True)\
    .option('inferSchema', True)\
    .load('abfss://bronze@datalakeadventure.dfs.core.windows.net/AdventureWorks_Sales_2017/AdventureWorks_Sales_2017.csv')

In [0]:
#all sales data into one df
df_Sales = spark.read.format('csv')\
    .option('header',True)\
    .option('inferSchema', True)\
    .load('abfss://bronze@datalakeadventure.dfs.core.windows.net/AdventureWorks_Sales*')

In [0]:
df_Territories = spark.read.format('csv')\
    .option('header',True)\
    .option('inferSchema', True)\
    .load('abfss://bronze@datalakeadventure.dfs.core.windows.net/AdventureWorks_Territories/AdventureWorks_Territories.csv')

## transformations

calender

In [0]:
df_calender = df_calender.withColumn('month', month(col('Date')))\
    .withColumn('year', year(col('Date'))) 


write to silver layer in PARQUET format

In [0]:
df_calender.write.format('parquet')\
    .mode('append')\
    .option('path', 'abfss://silver@datalakeadventure.dfs.core.windows.net/AdventureWorks_calender')\
    .save()

customers

In [0]:
df_Customers = df_Customers.withColumn('full_name', concat_ws(' ',col('Prefix'),col('FirstName'),col('LastName')))

In [0]:
df_Customers.write.format('parquet')\
    .mode('overwrite')\
    .option('path','abfss://silver@datalakeadventure.dfs.core.windows.net/AdventureWorks_Customers')\
    .save()

subcategories. no need of any transformation

In [0]:
df_Product_Subcategories.write.format('parquet')\
    .mode('overwrite')\
    .option('path','abfss://silver@datalakeadventure.dfs.core.windows.net/AdventureWorks_Subcategories')\
    .save()

products 

In [0]:
df_Products = df_Products.withColumn('ProductSKU',split(col('ProductSKU'),'-')[0])\
    .withColumn('ProductName',split(col("ProductName"),' ')[0])

In [0]:
df_Products.write.format('parquet')\
    .mode('overwrite')\
    .option('path','abfss://silver@datalakeadventure.dfs.core.windows.net/AdventureWorks_Products')\
    .save()

returns

In [0]:
df_Returns.write.format('parquet')\
    .mode('overwrite')\
    .option('path','abfss://silver@datalakeadventure.dfs.core.windows.net/AdventureWorks_Returns')\
    .save()

territories

In [0]:
%python
df_Territories.write.format('parquet')\
    .mode('overwrite')\
    .option('path','abfss://silver@datalakeadventure.dfs.core.windows.net/AdventureWorks_Territories')\
    .save()

sales

In [0]:
df_Sales = df_Sales.withColumn('StockDate', to_timestamp(col('StockDate')))\
    .withColumn('OrderNumber', regexp_replace(col('OrderNumber'),'S','T'))\
    .withColumn('multiply',col('OrderLineItem')*col('OrderQuantity'))

In [0]:
%python
df_Sales.write.format('parquet')\
    .mode('overwrite')\
    .option('path','abfss://silver@datalakeadventure.dfs.core.windows.net/AdventureWorks_Sales')\
    .save()

sales analysis

In [0]:
df_Sales.groupBy('OrderDate').agg(count(col('OrderNumber')).alias('total_order')).display()

product categories

In [0]:
df_Product_Categories.display()

Territories

In [0]:
df_Territories.write.format('parquet')\
    .mode('overwrite')\
    .option('path','abfss://silver@datalakeadventure.dfs.core.windows.net/AdventureWorks_Territories')\
    .save()